# ETA Prediction Model — Methodology Walkthrough

This notebook explains the design decisions, algorithms, and evolution of the
segmented ETA prediction model — from the initial prep-time-only approach through
the final v5.7.5 production model.

> **Note:** Production data is not included. This notebook walks through the
> methodology with code excerpts and synthetic examples to illustrate each concept.

## 1. Why Segmentation?

A monolithic model that predicts total delivery time from order creation treats the
entire journey as a black box. The problem: courier acceptance (T1), vendor prep (T2),
and transit (T3) have fundamentally different drivers and noise profiles.

| Segment | Primary Drivers | Noise Level |
|---------|----------------|-------------|
| T1: Courier acceptance | Supply/demand, time of day | Low — mostly deterministic |
| T2: Vendor pickup | Kitchen capacity, order complexity | High — inherently stochastic |
| T3: Transit | Distance, traffic, route | Medium — OSRM provides strong baseline |

By modeling each segment independently, we can:
- Use the right technique for each (ML for T1, ratio models for T2a/T3)
- Pinpoint where errors come from when debugging
- Improve one segment without risking regression in others

## 2. OSRM Enrichment Pipeline

Before modeling, every order needs OSRM-estimated driving times. The enrichment
pipeline (`osrm_enrichment.py`) handles this.

### Key design decisions:

**Coordinate validation and auto-fix:**
In the Saudi Arabia dataset, we discovered that ~2% of orders had lat/lon values
swapped (latitude > 40°, which is impossible for KSA). The pipeline automatically
detects and corrects this.

In [ ]:
# From osrm_enrichment.py — auto-fix swapped coordinates
# KSA latitudes are 16-33, longitudes are 34-57
# If lat > 40 and lon < 40, they're swapped

swap_cols = [
    ("pickup_lat", "pickup_lon"),
    ("dropoff_lat", "dropoff_lon"),
    ("courier_latitude_at_assignment", "courier_longitude_at_assignment")
]
for lat_c, lon_c in swap_cols:
    mask = (df[lat_c].abs() > 40) & (df[lon_c].abs() < 40)
    df.loc[mask, [lat_c, lon_c]] = df.loc[mask, [lon_c, lat_c]].values
    
print(f"Fixed {mask.sum()} swapped coordinate pairs")

**Route deduplication and parallel OSRM queries:**
Many orders share the same origin-destination pair (same restaurant → same neighbourhood).
Instead of querying OSRM for every order, we deduplicate coordinate pairs and query
unique routes only — reducing API calls by ~60%. Queries are parallelized with 16 threads.

## 3. Hierarchical Bayesian Smoothing

This is the core statistical technique used throughout the model. The problem:
we want ratio estimates at a fine granularity (zone × hour × distance bin), but
many cells have too few observations to be reliable.

**The solution: shrink sparse estimates toward broader aggregates.**

In [ ]:
import numpy as np

# The smoothing formula
def smoothed_ratio(n_cell, ratio_cell, n0, parent_ratio):
    """
    Empirical Bayes shrinkage.
    
    When n_cell is large, the cell's own ratio dominates.
    When n_cell is small, the parent (broader aggregate) dominates.
    n0 controls the crossover point.
    """
    return (n_cell * ratio_cell + n0 * parent_ratio) / (n_cell + n0)

# Example: A zone×hour×dist cell with n=5 orders and ratio=2.5
# vs the hour×dist parent with ratio=1.8

n0 = 60  # smoothing strength for T2a

# With 5 observations: heavily shrunk toward parent
print(f"n=5:   {smoothed_ratio(5, 2.5, n0, 1.8):.2f}")    # ~1.85

# With 50 observations: cell's own data carries more weight
print(f"n=50:  {smoothed_ratio(50, 2.5, n0, 1.8):.2f}")    # ~2.18

# With 200 observations: cell dominates, minimal shrinkage
print(f"n=200: {smoothed_ratio(200, 2.5, n0, 1.8):.2f}")   # ~2.44

**The parent hierarchy** cascades through increasingly broad levels:

```
zone × hour × dist_bin   (most specific)
       ↓ fallback
hour × dist_bin
       ↓ fallback
peak × dist_bin           (peak = hours 14-22)
       ↓ fallback
hour only
       ↓ fallback
peak only
       ↓ fallback
global median              (last resort)
```

This ensures **100% coverage** — every order gets a prediction, even for a
brand-new zone at an unusual hour.

## 4. Recency Weighting

Delivery patterns change over time: new restaurants open, courier fleet sizes
fluctuate, traffic patterns shift seasonally. Simple medians treat a 3-month-old
data point the same as yesterday's. Recency weighting fixes this.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Weight function: w = exp(-lambda * days_ago)
half_life = 21  # days
lam = np.log(2) / half_life

days_ago = np.arange(0, 90)
weights = np.exp(-lam * days_ago)

plt.figure(figsize=(10, 4))
plt.fill_between(days_ago, weights, alpha=0.3, color='#2196F3')
plt.plot(days_ago, weights, color='#2196F3', linewidth=2)
plt.axvline(x=21, color='red', linestyle='--', alpha=0.7, label='Half-life (21 days)')
plt.axvline(x=42, color='red', linestyle=':', alpha=0.5, label='2× half-life')
plt.xlabel('Days ago')
plt.ylabel('Weight')
plt.title('Recency Weight Decay (21-day half-life)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Today's observation:     weight = {weights[0]:.2f}")
print(f"21 days ago:             weight = {weights[21]:.2f}")
print(f"42 days ago:             weight = {weights[42]:.2f}")
print(f"90 days ago:             weight = {weights[89]:.4f}")

## 5. T2 Split: T2a + T2b

The v5.1 model treated the entire pickup phase as one segment. Analysis revealed
that it conflated two distinct processes:

- **T2a** (courier accepted → arrived at vendor): Primarily a driving problem.
  OSRM ratios work well here because it's mostly travel time.

- **T2b** (arrived at vendor → left vendor): Primarily a kitchen/waiting problem.
  This depends on the vendor, time of day, and order complexity — not distance.

Splitting T2 allowed us to use the right model for each sub-problem:
- T2a → OSRM ratio (same as T3)
- T2b → Shop × hour × day-of-week smoothed medians + prep-time priors

## 6. T2b Prep-Time Prior Blending

For T2b (vendor wait time), we have three tiers of prior information:

1. **Order-level priors** — prep times from individual historical orders
   (most granular: shop × vertical × hour × day-of-week)
2. **Shop-level priors** — aggregated prep times per restaurant
3. **Vertical-level priors** — category-wide averages (fast food, café, etc.)

These cascade: if order-level data exists for a shop, use it. If not, fall back
to shop-level. If that's missing, use the vertical average.

The final T2b prediction blends the cell estimate with both the parent and the
prep-time prior:

```python
blended = (n_cell × cell + N0_T2B × parent + N0_T2BPR × prior) / denominator
```

This three-source formula means even a brand-new restaurant gets a reasonable
T2b estimate from its cuisine vertical's historical patterns.

## 7. Coordinate Quality Assurance (T3)

GPS data from delivery apps is noisy. We found several failure modes:

In [ ]:
# From utils.py — flag_t3_coord_row()
# Each order's pickup/dropoff coordinates are checked for:

checks = {
    "zero_coords":          "GPS returned (0, 0) — device failure",
    "out_of_bbox":          "Coordinates outside KSA (16-33°N, 34-57°E)",
    "osrm_dist_missing":    "OSRM couldn't route this pair",
    "osrm_zero_vs_haversine": "OSRM says 0 km but haversine says >1 km",
    "hav/osrm_too_high":    "Straight-line distance >3× the routed distance",
    "hav/osrm_too_low":     "Straight-line distance <0.33× the routed distance",
}

print("T3 coordinate quality checks:")
for flag, desc in checks.items():
    print(f"  {flag:30s} → {desc}")

print("\nFlagged orders use zone×hour baseline instead of OSRM ratio.")

## 8. Model Evolution

The model went through multiple iterations. Key milestones:

| Version | What Changed | Impact |
|---------|-------------|--------|
| v1.0 | Prep-time model only | Couldn't predict full delivery time |
| v5.1 | Full T1+T2+T3+T4 segmented model | First end-to-end ETA |
| v5.1+ | Added hierarchical smoothing to OSRM ratios | Eliminated cold-start failures |
| v5.7 | Split T2 → T2a + T2b | Better vendor delay modeling |
| v5.7.5 | Recency weighting, coord QA, prep priors, bias correction | Final production model |

## 9. Results & Error Analysis

Validated on 137,081 orders with ground-truth timestamps.

### Where the errors come from:
- **T1** (MAE 0.9 min) — nearly solved; courier matching is predictable
- **T2** (MAE 6.9 min) — largest error source; kitchen delays are inherently noisy
- **T3** (MAE 3.8 min) — OSRM ratios work well; remaining error is last-mile

### 85.6% of predictions within 15 minutes of actual

The previous system showed a static 60-minute upper bound.
The model reduces median error to ~6 minutes — a fundamentally
different customer experience.

## 10. What I Would Do Differently

- **Drop-off coordinate quality** remains the biggest T3 issue. A geocoding
  validation layer against building polygons would catch more subtle errors.
- **T2b could benefit from real-time signals** — current kitchen queue length,
  recent prep times from the same restaurant — but this requires event streaming
  infrastructure the platform didn't have.
- **The bias correction is a patch.** A properly calibrated model shouldn't need
  post-hoc bias tables. Given more time, I'd investigate conformal prediction
  intervals for better uncertainty quantification.